# Notebook 47 — Option 2: `RestrictionEstimator`-assisted full-uniform-prior training

## Purpose

One of four notebooks (44/45/46/47) built 2026-09-06 to test remaining options for
salvaging System II fault classification. **You are running this notebook, not
Claude** — read the markdown cells, they explain what to watch for.

## The idea

The original nb31 posterior (full 5-D `box_uniform_5d()` prior) gave the best
classification result ever seen in this project: **macro-F1 = 0.694**. Its only later
failure (the 2026-08-27 "matched protocol" retrain) predates the Stage-1b warm-start
fix, so it's confounded with a since-fixed bug — genuinely unresolved whether the full
uniform prior would calibrate now. But a clean re-test this session
(`scripts/smoketest_sc2_fulluniform.py`) found it **prohibitively expensive**: 0/5
draws accepted, and — unlike every other prior tested — every single attempt (success
or fail) cost a uniform ~45s, meaning a full bank would take an estimated 50+ hours.

`sbi`'s `RestrictionEstimator` (Deistler et al. 2022) is built exactly for this
situation: train a small classifier to predict, from `theta` alone, whether a
simulation will be `valid` (no NaN/Inf) *before* paying for the expensive ODE solve.
Sampling from the resulting `restricted_prior` still costs one real, expensive
simulate() call per accepted draw — **the win is skipping the wasted expensive calls
on draws the classifier confidently predicts will fail**, which is exactly the observed
problem (every failure this session was just as expensive as a success).

## Honest caveat this notebook does not hide

**The bootstrap phase that trains the classifier itself needs real, expensive
labeled data** — there's no way around running the real simulator to know ground-truth
validity for the *initial* training set. This notebook's real go/no-go moment is
**Step 3**: a small, direct smoke test of the restricted prior's *real* simulator
accept rate. If that isn't a clear, substantial improvement over the historical ~0-37%,
**stop there** — per this project's established practice, don't blindly commit to a
full bank generation on a classifier that hasn't been shown to actually help.

## Design choice: bootstrap data is NOT reused in the final training bank

The classifier's `restrict_prior()` returns a well-defined prior object (support
restricted to the classifier's predicted-valid region), and the textbook-correct way to
train an SBI posterior against it is `SNPE(prior=restricted_prior)` with **fresh**
draws sampled from `restricted_prior` itself. Reusing the bootstrap phase's already-
accepted draws directly in the final bank would be more sample-efficient but introduces
a subtle bias (a real-valid draw the *classifier itself* would have rejected is not
actually a valid draw from `restricted_prior` as formally defined) — this notebook
keeps bootstrap and final-bank generation separate for correctness, at the cost of
some duplicated simulation effort. Note this if you want to optimize further later.


In [ ]:
import pathlib, pickle, sys, time

import numpy as np
import pandas as pd
import torch
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from sbi.utils import RestrictionEstimator
from cstr_sbi.recycle.priors import box_uniform_5d
from cstr_sbi.recycle.physics import PARAM_NAMES
from cstr_sbi.recycle.scenarios import list_closed_loop, FAULT_UNITS
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SC2
from sbi_pipeline import _structure_config, _simulate_summary

DATA = ROOT / "data"
RESULTS = ROOT / "results"
SBI_LOGS = ROOT / "sbi-logs"
for d in (DATA, RESULTS, SBI_LOGS):
    d.mkdir(exist_ok=True)

STRUCTURE = "S-C.2"
NOISE_PCT = 0.003
ARCH, HIDDEN, TRANSFORMS = "zuko_nsf", 60, 3
MAX_EPOCHS, STOP_AFTER_EPOCHS = 120, 12
SEED = 0
SUFFIX = "_fulluniform_restricted_stage1bfix"
N_TRAIN = 1000
N_SBC_SCREEN = 120
THRESH, Z0_NOM = 0.85, 0.90
N_REPS, N_POST = 30, 200

# --- Step 3 go/no-go threshold ---
# The historical (aborted, unreliable) accept rate for the raw full-uniform prior was
# ~37%; this session's own clean 5-draw smoke test got 0/5. Require the restricted
# prior to clear this bar by a wide margin before committing to a full bank.
MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE = 0.60

prior_uniform = box_uniform_5d()
print(f"Config: N_TRAIN={N_TRAIN}, N_SBC_SCREEN={N_SBC_SCREEN}, "
      f"go/no-go threshold={MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE:.0%}")


## Step 1 — bootstrap a labeled (valid/invalid) dataset for the classifier

Real, expensive simulation calls (~10-45s each per this session's smoke tests) — this
is the unavoidable upfront cost. Checkpointed every 50 rows (safe to interrupt/resume).
Failed draws are stored as a NaN-filled placeholder row so `RestrictionEstimator`'s
default `decision_criterion="nan"` labels them `invalid` automatically (matches this
project's own `s is None or not np.isfinite(s).all()` convention exactly).


In [ ]:
N_BOOTSTRAP = 250  # labeled draws to train the classifier on

def generate_bootstrap_labels(out_path, n_rows, seed):
    if out_path.exists():
        d = np.load(out_path)
        return (np.asarray(d["thetas"], dtype=np.float32), np.asarray(d["summaries"], dtype=np.float32))
    ctrl, y0 = _structure_config(STRUCTURE)
    rng = np.random.default_rng(seed)
    theta_rows, summary_rows = [], []
    n_valid = 0
    t0 = time.time()
    partial_path = out_path.with_suffix(".partial.npz")
    if partial_path.exists():
        p = np.load(partial_path)
        theta_rows = list(np.asarray(p["thetas"], dtype=np.float32))
        summary_rows = list(np.asarray(p["summaries"], dtype=np.float32))
        n_valid = int(p["n_valid"]) if "n_valid" in p.files else int(np.isfinite(summary_rows).all(axis=1).sum())
        print(f"resuming from partial: {len(theta_rows)}/{n_rows} rows ({n_valid} valid so far)")
    while len(theta_rows) < n_rows:
        th = prior_uniform.sample((1,)).numpy()[0]
        s = _simulate_summary(th, ctrl, y0, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
        if s is None or not np.isfinite(s).all():
            s = np.full(N_SUMMARIES_SC2, np.nan, dtype=np.float32)
        else:
            n_valid += 1
        theta_rows.append(th)
        summary_rows.append(s)
        if len(theta_rows) % 10 == 0:
            print(f"  {len(theta_rows)}/{n_rows}, valid={n_valid} "
                  f"(accept_rate={n_valid/len(theta_rows):.3f}), elapsed={time.time()-t0:.0f}s", flush=True)
        if len(theta_rows) % 50 == 0:
            np.savez(partial_path, thetas=np.asarray(theta_rows, dtype=np.float32),
                      summaries=np.asarray(summary_rows, dtype=np.float32), n_valid=n_valid)
    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    np.savez(out_path, thetas=theta_arr, summaries=summary_arr, n_valid=n_valid)
    print(f"saved {out_path}, wall_s={time.time()-t0:.0f}, "
          f"observed accept_rate={n_valid/len(theta_arr):.3f} ({n_valid}/{len(theta_arr)})")
    return theta_arr, summary_arr


theta_bootstrap, summary_bootstrap = generate_bootstrap_labels(
    DATA / f"wu2003_restriction_bootstrap_n{N_BOOTSTRAP}.npz", N_BOOTSTRAP, seed=20260906100)


## Step 2 — train the `RestrictionEstimator` classifier, build the restricted prior

`allowed_false_negatives=0.0` per the library's own documentation: "Inference is only
**exact** for `allowed_false_negatives=0.0`" — start exact; only loosen this later if
Step 3 shows the exact restriction is still too conservative (low accept rate).


In [ ]:
# NOTE: the default `stop_after_epochs=20` was found (this session's own pre-handoff
# validation, on a synthetic test) to sometimes under-train this classifier on a
# modestly-sized bootstrap set -- it can converge to "predict everything valid" before
# it has actually learned the boundary. Use a higher patience explicitly.
restriction_estimator = RestrictionEstimator(prior=prior_uniform)
restriction_estimator.append_simulations(torch.tensor(theta_bootstrap), torch.tensor(summary_bootstrap))
classifier = restriction_estimator.train(stop_after_epochs=40)
restricted_prior = restriction_estimator.restrict_prior(allowed_false_negatives=0.0)

# Cheap diagnostic (the CLASSIFIER's own predicted acceptance rate -- an optimistic
# proxy, NOT the real simulator's ground-truth rate; Step 3 below checks that directly).
_ = restricted_prior.sample((200,), save_acceptance_rate=True, show_progress_bars=False)
print(f"Classifier-predicted acceptance rate (proxy only, not yet validated against the "
      f"real simulator): {restricted_prior.acceptance_rate:.3f}")
if restricted_prior.acceptance_rate > 0.97:
    print("*** WARNING: this is suspiciously close to 1.0 -- given the bootstrap's own "
          "observed accept rate was well below that, this likely means the classifier "
          "under-fit (predicting almost everything 'valid') rather than genuinely finding "
          "a permissive boundary. Consider re-running this cell with a larger N_BOOTSTRAP "
          "or a higher stop_after_epochs before trusting Step 3's result. ***")


## Step 3 — GO/NO-GO: smoke-test the restricted prior against the REAL simulator

This is the moment that matters. Compare against:
- historical (unreliable, aborted) full-uniform accept rate: ~0.37
- this session's own clean 5-draw smoke test: 0/5 = 0.00
- `MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE` set above: 0.60

**If the real accept rate below doesn't clear the threshold, stop here** — do not run
the cells past this point. Report the result back rather than proceeding to a full
bank generation on an unproven classifier.


In [ ]:
N_SMOKE = 20
ctrl, y0 = _structure_config(STRUCTURE)
rng = np.random.default_rng(20260906200)
smoke_theta, smoke_summary = [], []
attempts = 0
t0 = time.time()
while len(smoke_theta) < N_SMOKE and attempts < 60:
    attempts += 1
    th = restricted_prior.sample((1,), show_progress_bars=False).numpy()[0]
    s = _simulate_summary(th, ctrl, y0, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
    if s is not None and np.isfinite(s).all():
        smoke_theta.append(th)
        smoke_summary.append(s)
    print(f"  attempt {attempts}: {'OK' if (s is not None and np.isfinite(s).all()) else 'FAIL'}  "
          f"accept_rate={len(smoke_theta)/attempts:.3f}  elapsed={time.time()-t0:.0f}s", flush=True)

restricted_accept_rate = len(smoke_theta) / attempts
print(f"\n=== STEP 3 RESULT: restricted-prior REAL accept rate = {restricted_accept_rate:.3f} "
      f"({len(smoke_theta)}/{attempts}) ===")
PROCEED = restricted_accept_rate >= MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE
if PROCEED:
    print(f"PASS -- clears the {MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE:.0%} threshold. "
          f"Proceeding to full bank generation below.")
else:
    print(f"FAIL -- does NOT clear the {MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE:.0%} threshold. "
          f"STOP HERE -- do not run the remaining cells. Report this result back instead "
          f"of committing to a full bank generation on an unproven classifier.")
    print("(You can manually override by setting PROCEED = True in this cell and re-running "
          "if you want to continue anyway, e.g. to test a lower MIN_ACCEPTABLE_RESTRICTED_ACCEPT_RATE.)")


## Step 4 (only if Step 3 passed) — generate the full training bank + one SBC screen set

Fresh draws from `restricted_prior`, not reusing Step 1's bootstrap data (see the
"Design choice" note at the top of this notebook). Checkpointed every 250 rows.


In [ ]:
def generate_bank(out_path, n_rows, seed, label):
    if out_path.exists():
        bank = np.load(out_path)
        print(f"[{label}] loading existing: {out_path}")
        return np.asarray(bank["thetas"], dtype=np.float32), np.asarray(bank["summaries"], dtype=np.float32)
    ctrl, y0 = _structure_config(STRUCTURE)
    rng = np.random.default_rng(seed)
    theta_rows, summary_rows = [], []
    t0 = time.time()
    attempts = 0
    partial_path = out_path.with_suffix(".partial.npz")
    if partial_path.exists():
        partial = np.load(partial_path)
        theta_rows = list(np.asarray(partial["thetas"], dtype=np.float32))
        summary_rows = list(np.asarray(partial["summaries"], dtype=np.float32))
        attempts = int(np.asarray(partial["attempts"]).item()) if "attempts" in partial.files else len(theta_rows)
        print(f"[{label}] resuming from partial: {len(theta_rows)}/{n_rows} rows, {attempts} attempts")
    while len(theta_rows) < n_rows:
        attempts += 1
        th = restricted_prior.sample((1,), show_progress_bars=False).numpy()[0]
        s = _simulate_summary(th, ctrl, y0, STRUCTURE, NOISE_PCT, rng, scenario_specific_warm_start=True)
        if s is None or not np.isfinite(s).all():
            continue
        theta_rows.append(th)
        summary_rows.append(s)
        if len(theta_rows) % 25 == 0:
            elapsed = time.time() - t0
            accept_rate = len(theta_rows) / attempts
            print(f"[{label}] {len(theta_rows)}/{n_rows}, attempts={attempts} "
                  f"(accept_rate={accept_rate:.3f}), elapsed={elapsed:.0f}s", flush=True)
        if len(theta_rows) % 250 == 0:
            np.savez(partial_path, thetas=np.asarray(theta_rows, dtype=np.float32),
                      summaries=np.asarray(summary_rows, dtype=np.float32), attempts=attempts)
    theta_arr = np.asarray(theta_rows, dtype=np.float32)
    summary_arr = np.asarray(summary_rows, dtype=np.float32)
    np.savez(out_path, thetas=theta_arr, summaries=summary_arr, attempts=attempts, seed=seed,
              structure=STRUCTURE, prior="restricted_full_uniform")
    print(f"[{label}] saved: {out_path}, wall_s={time.time()-t0:.0f}, "
          f"accept_rate={len(theta_arr)/attempts:.3f}")
    return theta_arr, summary_arr


if PROCEED:
    theta_train, summary_train = generate_bank(
        DATA / f"wu2003_sbi_train_sc2_true_n{N_TRAIN}{SUFFIX}.npz", N_TRAIN, seed=20260906300, label="train")
    theta_sbc, summary_sbc = generate_bank(
        RESULTS / f"sc2_sbc_set_n{N_SBC_SCREEN}{SUFFIX}.npz", N_SBC_SCREEN, seed=20260906301, label="sbc_screen")
else:
    print("Skipped (Step 3 did not pass and PROCEED was not manually overridden).")


## Step 5 (only if Step 3 passed) — train the SNPE posterior, `prior=restricted_prior`

Same recipe as every other S-C.2 posterior this session (`zuko_nsf`, 60/3,
`z_score_x='none'`). **Note the declared prior is `restricted_prior`, not
`box_uniform_5d()` directly** — this is the textbook-correct usage
(`RestrictionEstimator`'s whole point is that the resulting posterior is defined
w.r.t. the restricted prior, not the original one).


In [ ]:
if PROCEED:
    posterior_path = SBI_LOGS / f"wu2003_posterior_sc2_true_n{N_TRAIN}_zscorenone_seed{SEED}{SUFFIX}.pkl"
    if posterior_path.exists():
        with open(posterior_path, "rb") as f:
            posterior = pickle.load(f)["posterior"]
        print(f"Loaded existing posterior: {posterior_path}")
    else:
        torch.manual_seed(SEED)
        inference = SNPE(prior=restricted_prior, density_estimator=posterior_nn(
            model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS, z_score_x="none"))
        inference.append_simulations(torch.tensor(theta_train), torch.tensor(summary_train), proposal=restricted_prior)
        t0 = time.time()
        est = inference.train(max_num_epochs=MAX_EPOCHS, validation_fraction=0.1,
                                stop_after_epochs=STOP_AFTER_EPOCHS, show_train_summary=False)
        posterior = inference.build_posterior(est)
        print(f"trained in {time.time()-t0:.0f}s")
        with open(posterior_path, "wb") as f:
            pickle.dump({"posterior": posterior, "structure": "S-C.2", "n_train": N_TRAIN,
                          "torch_seed": SEED, "z_score_x": "none", "prior": "restricted_full_uniform"}, f)

    def run_sbc_screen(posterior, theta_set, summary_set, n_post, seed):
        torch.manual_seed(seed)
        ranks = {name: [] for name in PARAM_NAMES}
        for th, x_obs in zip(theta_set, summary_set):
            with torch.no_grad():
                samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
                                          show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
            for k, name in enumerate(PARAM_NAMES):
                ranks[name].append(int(np.sum(samp[:, k] < th[k])))
        out = {}
        for name in PARAM_NAMES:
            r = np.asarray(ranks[name], dtype=float)
            ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
            out[name] = float(ks.pvalue)
        return out

    sbc_result = run_sbc_screen(posterior, theta_sbc, summary_sbc, n_post=N_SBC_SCREEN, seed=500001)
    print("SBC screen (N=120, NOT a confirmation -- per finding_sbc_power_eta_col, N>=400 needed to trust a pass):")
    for name, p in sbc_result.items():
        print(f"  {name:10s} ks_pvalue={p:.4g}  {'PASS (screen)' if p > 0.05 else 'fail'}")


## Step 6 (only if Step 3 passed) — classification on the real 14-scenario catalogue

Same protocol as every other classification script this session.


In [ ]:
def sample_fault_unit(theta_row, thresh=THRESH, z0_nom=Z0_NOM):
    alpha, beta_r, eta_col, xi_reb, z_A0 = theta_row
    reactor_bad = (alpha < thresh) or (beta_r < thresh)
    column_bad = (eta_col < thresh) or (xi_reb < thresh)
    feed_bad = z_A0 < thresh * z0_nom
    n_bad = int(reactor_bad) + int(column_bad) + int(feed_bad)
    if n_bad == 0:
        return "healthy"
    if n_bad >= 2:
        return "multi"
    if reactor_bad:
        return "reactor"
    if column_bad:
        return "column"
    return "feed"


def classify_fault_recycle(samples):
    labels = [sample_fault_unit(row) for row in samples]
    probs = {u: float(np.mean([l == u for l in labels])) for u in FAULT_UNITS}
    return max(probs, key=probs.__getitem__), probs


def compute_classification_metrics(preds, trues, classes=FAULT_UNITS):
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1_per_class = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_per_class[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    macro_f1 = float(np.mean(list(f1_per_class.values())))
    accuracy = float(np.diag(cm).sum() / cm.sum()) if cm.sum() > 0 else 0.0
    return {"confusion_matrix": cm, "class_names": classes, "per_class_f1": f1_per_class,
            "macro_f1": macro_f1, "accuracy": accuracy}


if PROCEED:
    y0_sb = nominal_warm_start("S-B")
    rows = []
    for sc in list_closed_loop():
        theta_np = np.asarray(sc.theta(), dtype=np.float32)
        true_class = sc.fault_unit()
        t_h, raw_obs = deterministic_window(sc, structure="S-B", y0=y0_sb)
        t_h_arr = np.asarray(t_h)
        rng = np.random.default_rng(20260705)
        reps = noisy_replicates(raw_obs, n_replicates=N_REPS, rng=rng, noise_pct=NOISE_PCT)
        for i in range(N_REPS):
            s = compute_summaries(reps[i], STRUCTURE, t_h_arr, theta=theta_np, rng=rng)
            with torch.no_grad():
                samp = posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
                                          show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
            pred, probs = classify_fault_recycle(samp)
            rows.append({"scenario": sc.name, "replicate": i, "true_class": true_class, "pred_class": pred, **probs})
        print(f"  {sc.name:22s} true={true_class}")

    eval_df = pd.DataFrame(rows)
    eval_df.to_csv(RESULTS / "47_restricted_fulluniform_classification_per_replicate.csv", index=False)
    metrics = compute_classification_metrics(eval_df["pred_class"].tolist(), eval_df["true_class"].tolist())
    print(f"\n=== RestrictionEstimator-assisted full-uniform-prior classification ===")
    print(f"macro-F1={metrics['macro_f1']:.3f}  accuracy={metrics['accuracy']:.3f}")
    print("per-class F1:", {k: round(v, 3) for k, v in metrics['per_class_f1'].items()})
    np.savez(RESULTS / "47_restricted_fulluniform_classification_metrics.npz",
              cm=metrics["confusion_matrix"], macro_f1=metrics["macro_f1"], accuracy=metrics["accuracy"],
              class_names=FAULT_UNITS)


## Step 7 — context: compare against every other number this project has produced

| Method | macro-F1 |
|---|---|
| full-uniform-prior SBI (nb31, pre-bugfix, historical) | 0.694 |
| narrow-lognormal SBI (S-C.2 baseline) | 0.07-0.08 |
| alpha-only mixed-prior SBI | 0.11-0.112 |
| **RestrictionEstimator-assisted full-uniform SBI (this notebook)** | *(see Step 6 output above, if PROCEED)* |

**If this recovers something close to 0.694**, that's the single best outcome across
every option tried — it would mean the original nb31 result was real (not an artifact
of the now-fixed physics bug or the pre-warm-start-fix protocol) and the only blocker
was ever computational cost, now solved. **If it's meaningfully worse than 0.694 despite
a good Step 3 accept rate**, that would newly implicate either the physics bug fix
(Item 6) or the warm-start fix itself as having changed the underlying posterior's
behavior, not just its cost — worth flagging clearly either way when reporting back.


## Final acceptance criteria

| # | Criterion | Threshold | Rationale |
|---|---|---|---|
| 1 | Restricted-prior real accept rate (Step 3 go/no-go) | ≥ 60% | already gates whether Steps 4-6 run at all; restated here for a single final summary |
| 2 | SBC screen shows a calibration signal | min_ks_pvalue > 0.01 | a screen only (N=120, not a confirmation) — just checks the posterior isn't catastrophically miscalibrated |
| 3 | Classification macro-F1 vs. best-so-far | ≥ 0.11 | beats the best previously-achieved posterior-based result |
| 4 | Classification macro-F1 (stretch — recovering the historical result) | ≥ 0.50 | most of the way back to nb31's pre-bugfix 0.694 |


In [ ]:
criteria = [
    {"criterion": "1. Restricted-prior real accept rate >= 60%", "value": restricted_accept_rate,
     "threshold": 0.60, "pass": PROCEED},
]
if PROCEED:
    min_ks = sbc_result["min_ks_pvalue"] if "min_ks_pvalue" in sbc_result else min(sbc_result.values())
    macro_f1 = metrics["macro_f1"]
    criteria += [
        {"criterion": "2. SBC screen min_ks_pvalue > 0.01", "value": min_ks,
         "threshold": 0.01, "pass": min_ks > 0.01},
        {"criterion": "3. Classification macro-F1 >= 0.11", "value": macro_f1,
         "threshold": 0.11, "pass": macro_f1 >= 0.11},
        {"criterion": "4. Classification macro-F1 >= 0.50 (stretch)", "value": macro_f1,
         "threshold": 0.50, "pass": macro_f1 >= 0.50},
    ]
else:
    criteria += [
        {"criterion": "2. SBC screen min_ks_pvalue > 0.01", "value": float("nan"), "threshold": 0.01, "pass": False},
        {"criterion": "3. Classification macro-F1 >= 0.11", "value": float("nan"), "threshold": 0.11, "pass": False},
        {"criterion": "4. Classification macro-F1 >= 0.50 (stretch)", "value": float("nan"), "threshold": 0.50, "pass": False},
    ]

print("=" * 78)
print(f"{'Criterion':<50s} {'Value':>9s} {'Threshold':>10s} {'Result':>8s}")
print("-" * 78)
for c in criteria:
    val_str = f"{c['value']:.3f}" if c["value"] == c["value"] else "N/A"
    print(f"{c['criterion']:<50s} {val_str:>9s} {c['threshold']:>10.2f} {'PASS' if c['pass'] else 'FAIL':>8s}")
print("=" * 78)

if not PROCEED:
    print("\nVERDICT: STOPPED AT STEP 3 -- the restricted prior's real accept rate did not "
          "clear the go/no-go threshold. Criteria 2-4 could not be evaluated. This is the "
          "expected, honest outcome if RestrictionEstimator's classifier couldn't learn a "
          "useful boundary from this bootstrap size -- report the Step 3 accept rate back "
          "rather than the full-uniform-prior route being fully closed off.")
elif criteria[3]["pass"]:
    print("\nVERDICT: PASS (strong) -- recovers most of the historical nb31 result. This "
          "would be the single best outcome across every option tried this session.")
elif criteria[2]["pass"]:
    print("\nVERDICT: PASS (modest) -- beats every other option tried so far, but doesn't "
          "fully recover the historical ceiling. Worth reporting either way, and flag "
          "whether the physics-bug-fix or warm-start-fix themselves changed the posterior's "
          "behavior (not just its cost) if this is meaningfully below 0.694.")
else:
    print("\nVERDICT: FAIL -- the accept-rate problem was solved but classification still "
          "doesn't recover. This would newly implicate the physics-bug-fix or warm-start-fix "
          "as having changed the underlying posterior's behavior, not just made the full "
          "uniform prior expensive to train -- flag this clearly when reporting back.")
